<a href="https://colab.research.google.com/github/am-3/IB9AU-2026/blob/main/Task18_Synthetic_Data_Generation_and_Quality_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 18: Synthetic Data Generation and Quality Assessment
Atharva M
___
**Objective:** This notebook aims to demonstrate the process of generating synthetic tabular data using a Generative Adversarial Network (GAN) and rigorously evaluating its quality across three key pillars: Fidelity, Utility, and Privacy. The goal is to produce a synthetic dataset of 5,000 records from a real financial transaction dataset (`fraud_transactions.csv`) and assess how well it mimics the original data's statistical properties, supports downstream machine learning tasks, and preserves the privacy of individual records.
___
**Tech Stack:**
*   **Python:** The primary programming language.
*   **Pandas & NumPy:** For data manipulation and numerical operations.
*   **CTGAN (Conditional Tabular GAN):** A state-of-the-art generative model for synthetic tabular data, built on PyTorch.
*   **Scikit-learn:** For data preprocessing (OneHotEncoder, StandardScaler), model training (RandomForestClassifier), and evaluation metrics (roc_auc_score, f1_score).
*   **Matplotlib & Seaborn:** For data visualization.
*   **SciPy:** For statistical tests (Kolmogorov-Smirnov).
___
**Methodology:**
1.  **Data Loading & Preprocessing:** The `fraud_transactions.csv` dataset is loaded, and dynamically split into numerical and categorical features. A stratified train-test split is performed on the real data. Encoders and scalers are fitted exclusively on the real training data to prevent data leakage.
2.  **Synthetic Data Generation:** A CTGAN model is trained on the real training data. This model learns the underlying distribution of the data, including inter-column relationships. After training, 5,000 synthetic records are generated.
3.  **Quality Evaluation - Pillar 1: Fidelity:**
    *   **Univariate Fidelity:** Assessed using the Kolmogorov-Smirnov (KS) test to compare the distributions of individual numerical features between real and synthetic data.
    *   **Multivariate Fidelity:** Assessed by comparing the absolute differences between the correlation matrices of numerical features in the real and synthetic datasets.
4.  **Quality Evaluation - Pillar 2: Utility:**
    *   Evaluated using a "Train Synthetic, Test Real" (TSTR) approach. A Random Forest Classifier is trained on the synthetic data and its performance (AUC and F1-Score) is measured on the real test set. This is benchmarked against a "Train Real, Test Real" (TRTR) model trained on real data.
5.  **Quality Evaluation - Pillar 3: Privacy:**
    *   Assessed by measuring the distance of each synthetic record to its nearest neighbor in the real training data using k-Nearest Neighbors. Smaller distances can indicate potential memorization of real records.
    ___

## Pillar 1: Fidelity Evaluation - Detailed Analysis

Fidelity measures how well the synthetic data replicates the statistical properties of the real data. We assess this using both univariate and multivariate methods.

### A. Univariate Fidelity: Kolmogorov-Smirnov (KS) Test

The Kolmogorov-Smirnov (KS) test is a non-parametric test used to compare a sample with a reference probability distribution (one-sample KS test) or to compare two samples (two-sample KS test). Here, we use the two-sample KS test to quantify the difference between the empirical cumulative distribution functions (ECDFs) of each numerical feature in the real and synthetic datasets.

The KS statistic $D$ is defined as:

$$ D_n = \sup_x |F_n(x) - F(x)| $$

where $F_n(x)$ is the empirical distribution function of the real data, and $F(x)$ is the empirical distribution function of the synthetic data. A smaller $D$ value indicates a closer resemblance between the distributions. The $p$-value indicates the probability of observing such a difference by chance if the underlying distributions were identical. A low $p$-value (typically < 0.05) suggests that the distributions are significantly different.
___
## B. Multivariate Fidelity: Correlation Matrix Differences

Multivariate fidelity assesses how well the relationships between features are preserved in the synthetic data. We evaluate this by comparing the absolute differences between the correlation matrices of numerical features in the real and synthetic datasets. A lower mean absolute difference indicates better preservation of these relationships.

The Mean Absolute Difference (MAD) between correlation matrices $R_{real}$ and $R_{syn}$ is calculated as:

$$ \text{MAD} = \frac{1}{N^2} \sum_{i=1}^{N} \sum_{j=1}^{N} |(R_{real})_{ij} - (R_{syn})_{ij}| $$

where $N$ is the number of numerical features.
___
## Pillar 2: Utility Evaluation - Detailed Analysis

Utility assesses how well the synthetic data can be used for downstream machine learning tasks. We evaluate this using the "Train Synthetic, Test Real" (TSTR) approach and benchmark it against a "Train Real, Test Real" (TRTR) baseline. A Random Forest Classifier is used for this binary classification task (fraud detection).

### Metrics:

*   **Area Under the Receiver Operating Characteristic Curve (AUC-ROC):** A measure of a classifier's performance across all possible classification thresholds. It represents the probability that the model ranks a randomly chosen positive instance higher than a randomly chosen negative instance. Values range from 0 to 1, with 1 being a perfect classifier.

    $$ \text{AUC} = \int_0^1 \text{TPR}(FPR^{-1}(t)) dt $$

*   **F1-Score:** The harmonic mean of precision and recall. It is a good metric for imbalanced datasets, like fraud detection, as it considers both false positives and false negatives.

    $$ F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} $$

    where:
    *   Precision $= \frac{\text{True Positives}}{\text{True Positives} + \text{False Positives}}$
    *   Recall $= \frac{\text{True Positives}}{\text{True Positives} + \text{False Negatives}}$
___
## Pillar 3: Privacy Evaluation - Detailed Analysis

Privacy assessment, particularly approximate memorization, evaluates the risk of synthetic data leaking sensitive information from the real training data. We use a k-Nearest Neighbors (k-NN) approach for this. The core idea is to measure the distance from each synthetic record to its closest counterpart in the real training data.

### Metric: Nearest Neighbor Distance

We calculate the Euclidean distance (or other suitable distance metric) from each synthetic data point $s_j$ to its nearest neighbor $r_i$ in the real training dataset. Lower distances suggest that synthetic records are very similar to real records, indicating a higher risk of memorization or direct copying.

$$ \text{Distance}(s_j, r_i) = \min_{r \in \text{RealData}} ||s_j - r||_2 $$

We then analyze the distribution of these minimum distances. Ideally, synthetic data should have a distribution of nearest neighbor distances that indicates novelty, meaning synthetic points are not too close to any single real data point, especially for sensitive features.


In [ ]:
!pip install ctgan
import pandas as pd
import numpy as np
from ctgan import CTGAN
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score
from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors

# ==========================================
# 1. Load Data & Dynamic Preprocessing
# ==========================================
print("Loading data...")
# Load the dataset and drop missing values for simplicity
df_fraud = pd.read_csv('fraud_transactions.csv').dropna()

# --- IMPORTANT: Change this if your target column has a different name ---
target_col = 'is_fraud' # Common alternatives: 'Class', 'fraud', 'isFraud'

if target_col not in df_fraud.columns:
    # Fallback: guess the target column if 'is_fraud' isn't found
    possible_targets = ['Class', 'fraud', 'isFraud', 'target']
    for pt in possible_targets:
        if pt in df_fraud.columns:
            target_col = pt
            break

print(f"Using '{target_col}' as the target column.")

# Dynamically separate numerical and categorical columns
cat_cols = df_fraud.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
if target_col in cat_cols: cat_cols.remove(target_col)

num_cols = df_fraud.select_dtypes(include=[np.number]).columns.tolist()
if target_col in num_cols: num_cols.remove(target_col)

# Train/Test Split (Stratified to maintain fraud ratios)
real_train, real_test = train_test_split(
    df_fraud, test_size=0.2, random_state=42, stratify=df_fraud[target_col]
)

# ==========================================
# 2. Train CTGAN & Generate 5000 Records
# ==========================================
print("\nTraining CTGAN (this may take a few minutes)...")
discrete_cols = cat_cols + [target_col]

# Increased epochs to 200 and batch_size to 1000 for potentially better performance
ctgan = CTGAN(epochs=200, batch_size=1000, verbose=True)
ctgan.fit(real_train, discrete_columns=discrete_cols)

print("\nGenerating 5,000 synthetic records...")
synthetic_df = ctgan.sample(5000)

# ==========================================
# 3. Evaluate Pillar 1: Fidelity
# ==========================================
print("\n--- PILLAR 1: FIDELITY ---")
# A. Univariate (KS Test)
ks_results = {}
for col in num_cols:
    r = real_train[col].sample(min(5000, len(real_train)), random_state=42)
    s = synthetic_df[col].sample(min(5000, len(synthetic_df)), random_state=42)
    stat, pval = ks_2samp(r, s)
    ks_results[col] = {"ks_stat": stat, "p_value": pval}

ks_df = pd.DataFrame(ks_results).T.sort_values("ks_stat")
print("\nKolmogorov-Smirnov Test Results (Lower KS stat is better):")
display(ks_df.head())

# B. Multivariate (Correlation Matrix Differences)
real_corr = real_train[num_cols].corr()
syn_corr = synthetic_df[num_cols].corr()
corr_diff = (real_corr - syn_corr).abs().mean().mean()
print(f"\nMean Absolute Difference in Correlation Matrices: {corr_diff:.4f}")

# ==========================================
# 4. Evaluate Pillar 2: Utility (TSTR)
# ==========================================
print("\n--- PILLAR 2: UTILITY (TSTR vs TRTR) ---")

# Fit encoders/scalers ONLY on real_train to prevent leakage
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
scaler = StandardScaler()

if len(cat_cols) > 0: ohe.fit(real_train[cat_cols])
if len(num_cols) > 0: scaler.fit(real_train[num_cols])

def preprocess(df_subset):
    X_parts = []
    if len(cat_cols) > 0:
        X_parts.append(ohe.transform(df_subset[cat_cols]))
    if len(num_cols) > 0:
        X_parts.append(scaler.transform(df_subset[num_cols]))

    X_all = np.hstack(X_parts) if len(X_parts) > 1 else X_parts[0]
    y_out = df_subset[target_col].astype(int)
    return X_all, y_out

# Baseline: Train Real, Test Real (TRTR)
X_real_train, y_real_train = preprocess(real_train)
X_real_test, y_real_test = preprocess(real_test)

rf_real = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_real.fit(X_real_train, y_real_train)
y_proba_real = rf_real.predict_proba(X_real_test)[:, 1]
y_pred_real = (y_proba_real >= 0.5).astype(int)

auc_real = roc_auc_score(y_real_test, y_proba_real)
f1_real = f1_score(y_real_test, y_pred_real)

# Synthetic: Train Synthetic, Test Real (TSTR)
X_syn_train, y_syn_train = preprocess(synthetic_df)

rf_syn = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_syn.fit(X_syn_train, y_syn_train)
y_proba_syn = rf_syn.predict_proba(X_real_test)[:, 1]
y_pred_syn = (y_proba_syn >= 0.5).astype(int)

auc_syn = roc_auc_score(y_real_test, y_proba_syn)
f1_syn = f1_score(y_real_test, y_pred_syn)

utility_df = pd.DataFrame(
    {"AUC": [auc_real, auc_syn], "F1-Score": [f1_real, f1_syn]},
    index=["Baseline (Train REAL)", "Synthetic (Train CTGAN)"]
)
display(utility_df)

# ==========================================
# 5. Evaluate Pillar 3: Privacy
# ==========================================
print("\n--- PILLAR 3: PRIVACY (Approximate Memorization) ---")
# Sample down for speed
real_num = real_train[num_cols].sample(min(5000, len(real_train)), random_state=42)
syn_num = synthetic_df[num_cols].sample(min(5000, len(synthetic_df)), random_state=42)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(real_num)

distances, _ = nn.kneighbors(syn_num)
distances = distances.flatten()

privacy_df = pd.Series(distances).describe()
print("Nearest Neighbor Distances (Values close to 0.0 indicate memorization):")
print(privacy_df)


Loading data...
Using 'is_fraud' as the target column.

Training CTGAN (this may take a few minutes)...


Gen. (+00.08) | Discrim. (-00.00): 100%|██████████| 200/200 [06:45<00:00,  2.03s/it]



Generating 5,000 synthetic records...

--- PILLAR 1: FIDELITY ---

Kolmogorov-Smirnov Test Results (Lower KS stat is better):


,ks_stat,p_value
amt,0.0912,1.646764e-18



Mean Absolute Difference in Correlation Matrices: 0.0000

--- PILLAR 2: UTILITY (TSTR vs TRTR) ---


,AUC,F1-Score
Baseline (Train REAL),0.966999,0.83237
Synthetic (Train CTGAN),0.764981,0.00000



--- PILLAR 3: PRIVACY (Approximate Memorization) ---
Nearest Neighbor Distances (Values close to 0.0 indicate memorization):
count    5000.000000
mean        0.200082
std         0.814795
min         0.000001
25%         0.004290
50%         0.010981
75%         0.031792
max        14.741855
dtype: float64


## Insights & Technical Learnings

### Key Results:

*   **Fidelity:** The CTGAN model demonstrated mixed results in replicating the statistical properties of the real data. The Kolmogorov-Smirnov (KS) statistic for the 'amt' feature, which was the only numerical feature, was relatively low, suggesting a reasonable preservation of its univariate distribution. However, the exact value needs to be interpreted in context. The mean absolute difference in correlation matrices was 0.0000, which is an ideal result, indicating perfect preservation of numerical feature relationships. This could be due to having only one numerical feature ('amt'), making the correlation matrix trivial (a 1x1 matrix with a value of 1.0).

*   **Utility:** The synthetic data showed significantly lower utility for the fraud detection task compared to the real data. The model trained on real data (TRTR) achieved an AUC of **0.966999** and an F1-Score of **0.83237**. In contrast, the model trained on synthetic data (TSTR) achieved an AUC of **0.764981** and an F1-Score of **0**. This substantial drop indicates that while the synthetic data might capture some marginal distributions, it fails to capture the complex patterns necessary for effective fraud classification, especially concerning the highly imbalanced nature of fraud datasets. The F1-score for synthetic data is particularly low, suggesting a very poor ability to identify positive (fraudulent) cases.

*   **Privacy:** The privacy evaluation, based on nearest neighbor distances, shows a range of distances. The minimum distance is very close to 0, indicating that at least some synthetic records are almost identical to real records, posing a potential privacy risk due to memorization. The distribution and summary statistics of these distances need careful examination; a small mean or 25th percentile value could imply a higher risk of privacy leakage. A large maximum distance, however, could also suggest that some synthetic data points are quite novel, which is desirable.
___
### Technical Learnings:

*   **CTGAN Performance on Imbalanced Data:** While CTGAN is powerful for tabular data, its ability to generate high-utility synthetic data for highly imbalanced classification tasks (like fraud detection) requires careful tuning and potentially advanced sampling techniques or loss modifications. The significant drop in F1-score for the TSTR model highlights this challenge.
*   **Feature Engineering Importance:** The presence of only one numerical feature ('amt') might oversimplify the fidelity evaluation for correlation matrices. In real-world scenarios with multiple numerical features, a richer multivariate fidelity analysis would be possible. The dynamic preprocessing and handling of categorical and numerical features are robust.
*   **Computational Cost:** Training CTGAN models, especially with larger datasets and more epochs, can be computationally intensive and time-consuming. The 200 epochs used here are a reasonable starting point, but optimizing training parameters is crucial for efficiency.
*   **Interpretation of Privacy Metrics:** Nearest Neighbor distances provide a quantitative measure of memorization. However, interpreting these values requires domain expertise. What constitutes a 'safe' distance can vary depending on the sensitivity of the data and the application. Further analysis, such as comparing feature distributions for memorized vs. non-memorized synthetic points, could provide deeper insights.
___
### Practical Application:

*   **Data Augmentation for Imbalanced Datasets:** Despite the current utility limitations, synthetic data can still be valuable for augmenting imbalanced datasets, especially if combined with techniques that focus on generating high-quality minority class samples. This is crucial in fintech for fraud detection where real fraud instances are rare.
*   **Privacy-Preserving Data Sharing:** For scenarios where sharing real financial transaction data is restricted due to privacy regulations (e.g., GDPR, CCPA), synthetic data offers a promising alternative. However, the privacy evaluation highlights the need for robust techniques to minimize memorization, potentially through differential privacy mechanisms integrated into the GAN training.
*   **Model Development and Testing:** Synthetic data can accelerate the development and testing of new fraud detection models without requiring access to sensitive production data. This reduces reliance on data custodians and speeds up the development cycle, especially for prototyping and early-stage model iterations.
*   **Benchmarking and Research:** Synthetic data provides a controlled environment for benchmarking different machine learning algorithms and privacy-preserving techniques. It enables researchers to explore new methodologies without the complexities and constraints of real-world sensitive data.
___
